# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities in the dataset are referenced by their `@id` attributes.

In [ ]:
print("Record Sets:")
for rs in dataset.record_sets:
    print(f" - Name: {rs.name}; @id: {rs.id}")
    print("   Fields and Columns:")
    for field in rs.fields:
        print(f"     Field name: {field.name}; @id: {field.id}; column: {field.column.id if field.column else None}; dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

We'll extract all available record sets. For demonstration, we use the first record set found.

In [ ]:
# Extract data from all record sets
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Extracting records from record set @id {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns for record set '{record_set_id}': {dataframes[record_set_id].columns.tolist()}")
    print(f"Number of records: {len(dataframes[record_set_id])}\n")

# For further analysis, pick the first record set
if len(record_sets) > 0:
    first_record_set_id = record_sets[0]
    print(f"Sample records from record set '@id = {first_record_set_id}':")
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping data by attributes.

All fields are referenced by their `@id`.

In [ ]:
import numpy as np

# Use the first record set and pick a numeric field for demonstration
if len(record_sets) > 0:
    df = dataframes[first_record_set_id]

    # Identify numeric field by iterating fields and checking field.data_type
    numeric_field_id = None
    group_field_id = None
    for rs in dataset.record_sets:
        if rs.id == first_record_set_id:
            for field in rs.fields:
                # Pick the first numeric field (Integer or Float)
                if field.data_type in ["schema:Float", "schema:Integer", "http://schema.org/Float", "http://schema.org/Integer"]:
                    if field.column.id in df.columns:
                        numeric_field_id = field.column.id
                        break
            # Pick a group field (categorical), e.g., type Text
            for field in rs.fields:
                if field.data_type in ["schema:Text", "http://schema.org/Text", "dv:Text"]:
                    if field.column.id in df.columns:
                        group_field_id = field.column.id
                        break
            break

    if numeric_field_id is not None:
        # Filter for numeric_field > threshold
        # Attempt to infer threshold: for demonstration we use the 75th percentile
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        except Exception as e:
            print(f"Warning: could not convert field {numeric_field_id} to numeric: {e}")
        threshold = df[numeric_field_id].quantile(0.75) if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize
        if len(filtered_df) > 0:
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by group_field_id
            if group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
                display(grouped_df.head())
            else:
                print("No categorical group field found for grouping.")
        else:
            print("No records above threshold for numeric field.")
    else:
        print("No numeric field detected in the first record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the numeric field (by `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of numeric field (column @id: {numeric_field_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id is available, show a boxplot grouped by it
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Cannot plot: No suitable numeric field found.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and perform basic exploratory data analysis on a Croissant-described dataset using the `mlcroissant` library. All entities were referenced by their `@id` attributes for consistency and reproducibility.